<a href="https://colab.research.google.com/github/shahzadali1-git/DevelopersHub-AI-and-ML-Advanced-Tasks/blob/main/GRU_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [3]:
import pandas as pd
df=pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews'+"/IMDB Dataset.csv")
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})
y=df['sentiment'].values
y
#df.head()

array([1, 1, 1, ..., 0, 0, 0])

In [4]:
import re
def pre(text):
    text=text.lower()
    text=re.sub(r'<.*?>', '', text) # remove HTML tags
    text=re.sub(r'[^a-z]', ' ', text) # keep only letters
    text=re.sub(r'\s+', ' ', text) # remove extra spaces
    return text

df['review'] = df['review'].apply(pre)
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there s a family where a little boy ...,0
4,petter mattei s love in the time of money is a...,1
...,...,...
49995,i thought this movie did a down right good job...,1
49996,bad plot bad dialogue bad acting idiotic direc...,0
49997,i am a catholic taught in parochial elementary...,0
49998,i m going to have to disagree with the previou...,0


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

max_w=10000
len_sent=100

tok = Tokenizer(num_words=max_w, oov_token = '<oov>')
tok.fit_on_texts(df['review'])

seq = tok.texts_to_sequences(df['review'])

tok.word_index

{'<oov>': 1,
 'the': 2,
 'and': 3,
 'a': 4,
 'of': 5,
 'to': 6,
 'is': 7,
 'it': 8,
 'in': 9,
 'i': 10,
 'this': 11,
 'that': 12,
 's': 13,
 'was': 14,
 'as': 15,
 'movie': 16,
 'for': 17,
 'with': 18,
 'but': 19,
 'film': 20,
 'you': 21,
 't': 22,
 'on': 23,
 'not': 24,
 'he': 25,
 'are': 26,
 'his': 27,
 'have': 28,
 'one': 29,
 'be': 30,
 'all': 31,
 'at': 32,
 'they': 33,
 'by': 34,
 'an': 35,
 'who': 36,
 'so': 37,
 'from': 38,
 'like': 39,
 'there': 40,
 'or': 41,
 'just': 42,
 'her': 43,
 'out': 44,
 'about': 45,
 'if': 46,
 'has': 47,
 'what': 48,
 'some': 49,
 'good': 50,
 'can': 51,
 'when': 52,
 'more': 53,
 'very': 54,
 'she': 55,
 'up': 56,
 'no': 57,
 'time': 58,
 'my': 59,
 'even': 60,
 'would': 61,
 'which': 62,
 'only': 63,
 'story': 64,
 'really': 65,
 'see': 66,
 'their': 67,
 'had': 68,
 'me': 69,
 'well': 70,
 'we': 71,
 'were': 72,
 'than': 73,
 'much': 74,
 'bad': 75,
 'get': 76,
 'been': 77,
 'other': 78,
 'do': 79,
 'people': 80,
 'great': 81,
 'will': 82,
 'al

In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
pad=pad_sequences(seq,maxlen=100,padding='post')
pad

array([[  14, 2219,   10, ...,  127, 4058,  480],
       [9578,   34,    2, ..., 1952,   70,  221],
       [  14, 3017,   12, ...,   66,   18,  349],
       ...,
       [   2,   89, 1189, ...,    1,    3, 5907],
       [ 135,   33,  266, ...,   51,  727,   45],
       [ 687,  474,   11, ...,  784,   11,   16]], dtype=int32)

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Embedding, Dropout, Input

model = Sequential([
    Input(shape=(100,)),

    # Embedding with padding handling
    Embedding(input_dim=10000, output_dim=128, mask_zero=True),

    # GRU layer (main sequence learner)
    GRU(64),

    # Regularization
    Dropout(0.3),

    # Dense layers (clean and sufficient)
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(16, activation='relu'),

    # Output layer
    Dense(1, activation='sigmoid')
])

# Compile properly
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,322,465 (5.04 MB)

 Trainable params: 1,322,465 (5.04 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(pad,y,test_size=0.2,random_state=42)
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=3,batch_size=64)

Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 97s 150ms/step - accuracy: 0.7868 - loss: 0.4393 - val_accuracy: 0.8534 - val_loss: 0.3364
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 93s 149ms/step - accuracy: 0.8863 - loss: 0.2815 - val_accuracy: 0.8667 - val_loss: 0.3086
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 98s 156ms/step - accuracy: 0.9186 - loss: 0.2073 - val_accuracy: 0.8599 - val_loss: 0.3623


In [15]:
new = input('enter any comment: ')
new = pre(new)
new = tok.texts_to_sequences([new])
new = pad_sequences(new,maxlen=max_w)

prediction = model.predict(new)
print(prediction)
if prediction > 0.5:
    print('positive')
else:
    print('negative')

enter any comment: this movie was amazing
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 606ms/step
[[0.8738645]]
positive
